# NanoGPT ROCStories — Demo 3 (Final Submission)

**COMP4680/8650 Advanced ML Topics**

This notebook trains and evaluates 4 model variants on the ROCStories dataset:
- **M1 — Forward** (Task 1 baseline, SUBMISSION MODEL)
- **M2 — Reverse** (Task 2 experiment)
- **M3 — Structured** (Task 2 experiment)
- **M4 — LLaMA** (Task 2 novelty: RMSNorm + SwiGLU)

**Research question:** Does narrative direction/structure matter to an LM?

Demo 3 fixes from Demo 2:
- `block_size=512` (was 256) — more context per sample
- `dropout=0.3` (was 0.2) — stronger regularisation
- `max_iters=5000` (was 10000) — stops before overfitting
- `weight_decay=0.15` (was 0.1) — better generalisation
- `batch_size=32` with `gradient_accumulation_steps=2` — handles larger block_size
- `eval_iters=200` (was 100) — more stable val loss estimate

**Target: M1 Val PPL < 23, Eval PPL < 25.0**

## Step 0: GPU Check + Install Dependencies + Mount Drive

In [ ]:
# Check GPU availability
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected! Go to Runtime > Change runtime type > GPU')

# Install dependencies
!pip install tiktoken datasets nltk pandas matplotlib huggingface_hub -q

# Download NLTK data
import nltk
nltk.download('punkt_tab', quiet=True)

# Mount Google Drive for persistent checkpoint storage
from google.colab import drive
drive.mount('/content/drive')

# Create base directory on Drive
import os
DRIVE_BASE = '/content/drive/MyDrive/nanoGPT_demo3'
os.makedirs(DRIVE_BASE, exist_ok=True)
print(f'Drive base: {DRIVE_BASE}')
print('Step 0 complete!')

## Step 1: Clone Repository

In [ ]:
import subprocess
import os

REPO_URL = 'https://github.com/ibraKH/nanoGPT.git'
CLONE_DIR = '/content/nanoGPT'
WORK_DIR = '/content/nanoGPT/demo3'

# Option A: Clone from GitHub
def clone_repo():
    """Clone the repo with proper error handling."""
    if os.path.exists(CLONE_DIR):
        print(f'Removing existing {CLONE_DIR}...')
        subprocess.run(['rm', '-rf', CLONE_DIR], check=True)

    # Support GITHUB_TOKEN for private repos
    url = REPO_URL
    token = os.environ.get('GITHUB_TOKEN', '')
    if token:
        url = REPO_URL.replace('https://', f'https://{token}@')
        print('Using GITHUB_TOKEN for authentication')

    print(f'Cloning from {REPO_URL}...')
    result = subprocess.run(
        ['git', 'clone', '--depth', '1', url, CLONE_DIR],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f'ERROR: git clone failed (exit code {result.returncode})')
        print(f'stderr: {result.stderr}')
        return False
    print('Clone successful!')
    return True

# Try cloning
clone_ok = clone_repo()

if not clone_ok:
    print('\n--- Option B: Upload zip manually ---')
    print('1. Download your repo as a ZIP from GitHub')
    print('2. Upload it using the file upload below')
    print('3. Re-run this cell after uploading')
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        zip_name = list(uploaded.keys())[0]
        subprocess.run(['unzip', '-o', zip_name, '-d', '/content/'], check=True)
        # Find the extracted directory
        for d in os.listdir('/content/'):
            if d.startswith('nanoGPT') and os.path.isdir(f'/content/{d}'):
                if d != 'nanoGPT':
                    os.rename(f'/content/{d}', CLONE_DIR)
                break
        print('Zip extraction complete!')

# Verify required files
os.chdir(WORK_DIR)
required_files = [
    'model.py', 'model_llama.py', 'train.py', 'train_llama.py',
    'sample.py', 'sample_llama.py', 'eval.py', 'configurator.py',
    'evaluate_local.py',
    'data/rocstories/prepare.py',
    'data/rocstories/eval_stories.txt',
    'data/rocstories/eval_prompts.txt',
    'config/train_m1.py', 'config/train_m2_reverse.py',
    'config/train_m3_structured.py', 'config/train_m4_llama.py',
    'config/test_m1.py', 'config/test_m2_reverse.py',
    'config/test_m3_structured.py', 'config/test_m4_llama.py',
]
print(f'\nWorking directory: {os.getcwd()}')
all_present = True
for f in required_files:
    exists = os.path.exists(f)
    status = 'OK' if exists else 'MISSING'
    if not exists:
        all_present = False
    print(f'  [{status}] {f}')

if all_present:
    print('\nAll required files present!')
else:
    print('\nWARNING: Some files are missing!')

print('Step 1 complete!')

## Step 2: Symlink Output Directories to Google Drive

In [ ]:
import os

os.chdir(WORK_DIR)

# Checkpoint directories to symlink to Drive
out_dirs = [
    'out-m1', 'out-m2-reverse', 'out-m3-structured', 'out-m4-llama',
    'out-test-m1', 'out-test-m2', 'out-test-m3', 'out-test-m4',
]

DRIVE_BASE = '/content/drive/MyDrive/nanoGPT_demo3'

for d in out_dirs:
    drive_path = os.path.join(DRIVE_BASE, d)
    local_path = os.path.join(WORK_DIR, d)

    # Create directory on Drive
    os.makedirs(drive_path, exist_ok=True)

    # Remove existing local dir/symlink if any
    if os.path.islink(local_path):
        os.unlink(local_path)
    elif os.path.isdir(local_path):
        import shutil
        shutil.rmtree(local_path)

    # Create symlink
    os.symlink(drive_path, local_path)
    print(f'  {d} -> {drive_path}')

# Also create results dir
os.makedirs('results/samples', exist_ok=True)

print('\nSymlinks created. Checkpoints will persist on Google Drive.')
print('Step 2 complete!')

## Step 3: Download + Parse ROCStories Dataset

In [ ]:
import os
os.chdir(WORK_DIR)

# Load the dataset from HuggingFace
from datasets import load_dataset
import pandas as pd
import nltk
from nltk.tokenize import sent_tokenize

print('Loading ROCStories from HuggingFace (mintujupally/ROCStories)...')
ds = load_dataset('mintujupally/ROCStories', split='train')
df = ds.to_pandas()
print(f'Loaded {len(df)} stories')
print(f'Columns: {list(df.columns)}')

# Check column format
if 'sentence1' in df.columns:
    print('Dataset has pre-parsed sentence columns')
    # Preview
    for i in range(1, 6):
        col = 'sentence' + str(i)
        print(f'  {col}: {df[col].iloc[0]}')
elif 'text' in df.columns:
    print('Dataset has text column, parsing with NLTK...')
    print(f'  Sample text: {df["text"].iloc[0][:200]}')

    # Parse into 5 sentences
    rows = []
    for idx, row in df.iterrows():
        text = str(row['text']).strip()
        if not text:
            continue
        sents = sent_tokenize(text)
        if len(sents) < 5:
            sents = sents + [''] * (5 - len(sents))
        elif len(sents) > 5:
            sents = sents[:5]
        rows.append({
            'sentence1': sents[0],
            'sentence2': sents[1],
            'sentence3': sents[2],
            'sentence4': sents[3],
            'sentence5': sents[4],
        })

    df = pd.DataFrame(rows)
    csv_path = 'data/rocstories/rocstories_parsed.csv'
    df.to_csv(csv_path, index=False)
    print(f'Parsed {len(df)} stories, saved to {csv_path}')

    # Preview parsed sentences
    for i in range(1, 6):
        col = 'sentence' + str(i)
        print(f'  {col}: {df[col].iloc[0]}')
else:
    raise ValueError(f'Unexpected columns: {list(df.columns)}')

print(f'\nTotal stories: {len(df)}')

# Run prepare.py for all variants
print('\n--- Preparing all data variants ---')
!python data/rocstories/prepare.py --variant=all --seed=42

# Verify output files
print('\n--- Verifying data files ---')
for ddir in ['data/rocstories', 'data/rocstories_reverse', 'data/rocstories_struct']:
    for fname in ['train.bin', 'val.bin', 'meta.pkl']:
        fpath = os.path.join(ddir, fname)
        if os.path.exists(fpath):
            size_mb = os.path.getsize(fpath) / (1024 * 1024)
            print(f'  [OK] {fpath} ({size_mb:.2f} MB)')
        else:
            print(f'  [MISSING] {fpath}')

print('\nStep 3 complete!')

## Step 4: Smoke Tests (50 iters each)

In [ ]:
import os
import time
os.chdir(WORK_DIR)

smoke_tests = [
    ('M1 Forward',    'python train.py config/test_m1.py'),
    ('M2 Reverse',    'python train.py config/test_m2_reverse.py'),
    ('M3 Structured', 'python train.py config/test_m3_structured.py'),
    ('M4 LLaMA',      'python train_llama.py config/test_m4_llama.py'),
]

for name, cmd in smoke_tests:
    print(f'\n{"=" * 50}')
    print(f'  Smoke test: {name}')
    print(f'{"=" * 50}')
    t0 = time.time()
    !{cmd}
    elapsed = time.time() - t0
    print(f'\n  {name} smoke test completed in {elapsed:.1f}s')

print('\nAll smoke tests complete!')
print('Step 4 complete!')

## Step 5: Full Training (5000 iters each)

In [ ]:
import os
import time
os.chdir(WORK_DIR)

full_training = [
    ('M1 Forward (SUBMISSION)', 'python train.py config/train_m1.py',        'out-m1'),
    ('M2 Reverse',             'python train.py config/train_m2_reverse.py',  'out-m2-reverse'),
    ('M3 Structured',          'python train.py config/train_m3_structured.py','out-m3-structured'),
    ('M4 LLaMA',               'python train_llama.py config/train_m4_llama.py','out-m4-llama'),
]

timings = {}
for name, cmd, out_dir in full_training:
    print(f'\n{"=" * 60}')
    print(f'  Training: {name}')
    print(f'{"=" * 60}')
    t0 = time.time()
    !{cmd}
    elapsed = time.time() - t0
    timings[name] = elapsed
    print(f'\n  {name} completed in {elapsed/60:.1f} min ({elapsed:.0f}s)')

    # Check checkpoint
    ckpt_path = os.path.join(out_dir, 'ckpt.pt')
    if os.path.exists(ckpt_path):
        size_mb = os.path.getsize(ckpt_path) / (1024 * 1024)
        print(f'  Checkpoint: {ckpt_path} ({size_mb:.1f} MB)')
    else:
        print(f'  WARNING: No checkpoint saved at {ckpt_path}')

# Summary
print(f'\n{"=" * 60}')
print('  TRAINING SUMMARY')
print(f'{"=" * 60}')
total = 0
for name, elapsed in timings.items():
    print(f'  {name:<30} {elapsed/60:>6.1f} min')
    total += elapsed
print(f'  {"TOTAL":<30} {total/60:>6.1f} min')
print('\nStep 5 complete!')

## Step 6: Evaluation — Compute PPL for All Models

In [ ]:
import os
import math
import torch
import tiktoken
from contextlib import nullcontext
os.chdir(WORK_DIR)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16'
ptdtype = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[dtype]
ctx = nullcontext() if device == 'cpu' else torch.amp.autocast(device_type='cuda', dtype=ptdtype)

enc = tiktoken.get_encoding('gpt2')
encode = lambda s: enc.encode(s, allowed_special={'<|endoftext|>'})

# Load eval stories
with open('data/rocstories/eval_stories.txt', 'r') as f:
    content = f.read()
stories = [p.strip() for p in content.split('\n\n') if p.strip()]
print(f'Loaded {len(stories)} eval stories')

def load_model_for_eval(ckpt_dir, model_type='gpt'):
    ckpt_path = os.path.join(ckpt_dir, 'ckpt.pt')
    if not os.path.exists(ckpt_path):
        print(f'  No checkpoint at {ckpt_path}')
        return None
    checkpoint = torch.load(ckpt_path, map_location=device)
    if model_type == 'llama':
        from model_llama import GPTConfig, GPTLlama
        conf = GPTConfig(**checkpoint['model_args'])
        model = GPTLlama(conf)
    else:
        from model import GPTConfig, GPT
        conf = GPTConfig(**checkpoint['model_args'])
        model = GPT(conf)
    state_dict = checkpoint['model']
    for k in list(state_dict.keys()):
        if k.startswith('_orig_mod.'):
            state_dict[k[len('_orig_mod.'):]] = state_dict.pop(k)
    model.load_state_dict(state_dict)
    model.eval()
    model.to(device)
    return model

def compute_ppl(model, texts):
    total_nll = 0.0
    total_tokens = 0
    block_size = model.config.block_size
    with torch.no_grad():
        with ctx:
            for text in texts:
                ids = encode(text)
                if len(ids) < 2:
                    continue
                pos = 0
                while pos < len(ids) - 1:
                    inp = ids[pos:pos + block_size]
                    tgt = ids[pos + 1:pos + 1 + block_size]
                    if not tgt:
                        break
                    if len(inp) != len(tgt):
                        inp = inp[:len(tgt)]
                    x = torch.tensor(inp, dtype=torch.long, device=device)[None, :]
                    y = torch.tensor(tgt, dtype=torch.long, device=device)[None, :]
                    _, loss = model(x, y)
                    total_nll += loss.item() * len(tgt)
                    total_tokens += len(tgt)
                    pos += len(tgt)
    return math.exp(total_nll / total_tokens) if total_tokens > 0 else float('inf')

def format_reverse(stories):
    from nltk.tokenize import sent_tokenize
    result = []
    for s in stories:
        sents = sent_tokenize(s)
        if len(sents) < 5:
            sents += [''] * (5 - len(sents))
        elif len(sents) > 5:
            sents = sents[:5]
        result.append(' '.join([x for x in reversed(sents) if x.strip()]))
    return result

def format_structured(stories):
    from nltk.tokenize import sent_tokenize
    result = []
    for s in stories:
        sents = sent_tokenize(s)
        if len(sents) < 5:
            sents += [''] * (5 - len(sents))
        elif len(sents) > 5:
            sents = sents[:5]
        parts = [f'<S{i+1}> {x.strip()}' for i, x in enumerate(sents) if x.strip()]
        result.append(' '.join(parts))
    return result

# Evaluate all models
models_config = [
    ('M1-Forward',    'out-m1',            'forward', 'gpt'),
    ('M2-Reverse',    'out-m2-reverse',    'reverse', 'gpt'),
    ('M3-Structured', 'out-m3-structured', 'structured', 'gpt'),
    ('M4-LLaMA',      'out-m4-llama',      'llama', 'llama'),
]

ppl_results = {}
for name, ckpt_dir, variant, mtype in models_config:
    print(f'\n--- {name} ---')
    model = load_model_for_eval(ckpt_dir, mtype)
    if model is None:
        ppl_results[name] = float('inf')
        continue
    if variant == 'reverse':
        eval_texts = format_reverse(stories)
    elif variant == 'structured':
        eval_texts = format_structured(stories)
    else:
        eval_texts = stories
    ppl = compute_ppl(model, eval_texts)
    ppl_results[name] = ppl
    print(f'  Eval PPL: {ppl:.2f}')
    if name == 'M1-Forward':
        status = 'PASS' if ppl <= 25.0 else 'FAIL'
        print(f'  >>> {status} (PPL {ppl:.2f} vs threshold 25.0)')
    del model
    torch.cuda.empty_cache()

# Print results table
print(f'\n{"=" * 50}')
print(f'{"Model":<20} {"PPL":>10} {"Status":>10}')
print('-' * 42)
for name in ['M1-Forward', 'M2-Reverse', 'M3-Structured', 'M4-LLaMA']:
    ppl = ppl_results.get(name, float('inf'))
    status = ''
    if name == 'M1-Forward':
        status = 'PASS' if ppl <= 25.0 else 'FAIL'
    print(f'{name:<20} {ppl:>10.2f} {status:>10}')

# Hypothesis results
m1 = ppl_results.get('M1-Forward', float('inf'))
m2 = ppl_results.get('M2-Reverse', float('inf'))
m3 = ppl_results.get('M3-Structured', float('inf'))
m4 = ppl_results.get('M4-LLaMA', float('inf'))

print(f'\nHYPOTHESIS RESULTS:')
if m1 < float('inf') and m2 < float('inf'):
    h1 = 'SUPPORTED' if m2 > m1 else 'NOT SUPPORTED'
    print(f'  H1 (M2 > M1, causal asymmetry): {h1} (M1={m1:.2f}, M2={m2:.2f})')
if m1 < float('inf') and m3 < float('inf'):
    h2 = 'SUPPORTED' if m3 < m1 else 'NOT SUPPORTED'
    print(f'  H2 (M3 < M1, structure helps): {h2} (M1={m1:.2f}, M3={m3:.2f})')
if m1 < float('inf') and m4 < float('inf'):
    diff_pct = abs(m4 - m1) / m1 * 100
    if m4 < m1:
        h3 = 'LLaMA BETTER'
    elif diff_pct < 5:
        h3 = 'APPROX EQUAL (data bottleneck)'
    else:
        h3 = 'GPT-2 BETTER'
    print(f'  H3 (LLaMA vs GPT-2): {h3} (M1={m1:.2f}, M4={m4:.2f}, diff={diff_pct:.1f}%)')

print('\nStep 6 complete!')

## Step 7: Generate Samples

In [ ]:
import os
import torch
import tiktoken
from contextlib import nullcontext
os.chdir(WORK_DIR)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16'
ptdtype = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[dtype]
ctx = nullcontext() if device == 'cpu' else torch.amp.autocast(device_type='cuda', dtype=ptdtype)

enc = tiktoken.get_encoding('gpt2')
encode = lambda s: enc.encode(s, allowed_special={'<|endoftext|>'})
decode = lambda l: enc.decode(l)

# Load prompts
with open('data/rocstories/eval_prompts.txt', 'r') as f:
    prompts = [line.strip() for line in f if line.strip()]
print(f'Loaded {len(prompts)} eval prompts')

def generate_text(model, start_text, max_new_tokens=200, temperature=0.8, top_k=100):
    ids = encode(start_text)
    x = torch.tensor(ids, dtype=torch.long, device=device)[None, ...]
    with torch.no_grad():
        with ctx:
            y = model.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)
    return decode(y[0].tolist())

os.makedirs('results/samples', exist_ok=True)

# Generate for each model
sample_configs = [
    ('M1-Forward',    'out-m1',            'gpt',   '\n'),
    ('M2-Reverse',    'out-m2-reverse',    'gpt',   '\n'),
    ('M3-Structured', 'out-m3-structured', 'gpt',   '<S1>'),
    ('M4-LLaMA',      'out-m4-llama',      'llama', '\n'),
]

for name, ckpt_dir, mtype, start in sample_configs:
    print(f'\n{"=" * 50}')
    print(f'  Generating samples: {name}')
    print(f'{"=" * 50}')

    model = load_model_for_eval(ckpt_dir, mtype)
    if model is None:
        continue

    # Generate 5 free samples
    all_samples = []
    for i in range(5):
        text = generate_text(model, start)
        all_samples.append(text)
        print(f'\n--- Sample {i+1} ---')
        print(text[:300])

    safe_name = name.lower().replace('-', '_')
    with open(f'results/samples/{safe_name}_samples.txt', 'w') as f:
        for i, s in enumerate(all_samples):
            f.write(f'--- Sample {i+1} ---\n{s}\n\n')

    # M3: guided completions from prompts
    if name == 'M3-Structured':
        print(f'\n--- Guided completions for {name} ---')
        guided = []
        for prompt in prompts:
            guided_start = f'<S1> {prompt}'
            text = generate_text(model, guided_start)
            guided.append(text)
            print(f'\nPrompt: {prompt}')
            print(f'Output: {text[:200]}')

        with open(f'results/samples/{safe_name}_guided.txt', 'w') as f:
            for i, s in enumerate(guided):
                f.write(f'--- Guided {i+1} (prompt: {prompts[i]}) ---\n{s}\n\n')

    del model
    torch.cuda.empty_cache()

print('\nStep 7 complete!')

## Step 8: Sampling Parameter Sweep for M1 (CRITICAL for Qwen Score)

In [ ]:
import os
import torch
import tiktoken
from contextlib import nullcontext
os.chdir(WORK_DIR)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16'
ptdtype = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[dtype]
ctx = nullcontext() if device == 'cpu' else torch.amp.autocast(device_type='cuda', dtype=ptdtype)

enc = tiktoken.get_encoding('gpt2')
encode = lambda s: enc.encode(s, allowed_special={'<|endoftext|>'})
decode = lambda l: enc.decode(l)

# Load prompts
with open('data/rocstories/eval_prompts.txt', 'r') as f:
    prompts = [line.strip() for line in f if line.strip()]

# Load M1 model
print('Loading M1 model for sampling sweep...')
model = load_model_for_eval('out-m1', 'gpt')
assert model is not None, 'M1 checkpoint not found!'

def gen(start_text, max_new_tokens=200, temperature=0.8, top_k=100):
    ids = encode(start_text)
    x = torch.tensor(ids, dtype=torch.long, device=device)[None, ...]
    with torch.no_grad():
        with ctx:
            y = model.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)
    return decode(y[0].tolist())

# Sweep grid
temperatures = [0.7, 0.8, 0.9]
top_ks = [50, 100, 200]

print(f'Sweep: temperature x top_k')
print(f'Temperatures: {temperatures}')
print(f'Top-k values: {top_ks}')
print(f'Using {len(prompts[:3])} prompts per combo\n')

sweep_results = []
for temp in temperatures:
    for tk in top_ks:
        print(f'\n{"=" * 60}')
        print(f'  temperature={temp}, top_k={tk}')
        print(f'{"=" * 60}')

        combo_samples = []
        for prompt in prompts[:3]:
            text = gen(prompt, max_new_tokens=200, temperature=temp, top_k=tk)
            combo_samples.append(text)
            print(f'\n  Prompt: {prompt}')
            print(f'  Output: {text[:250]}')

        sweep_results.append({
            'temperature': temp,
            'top_k': tk,
            'samples': combo_samples
        })

        # Save
        with open(f'results/samples/m1_sweep_t{temp}_k{tk}.txt', 'w') as f:
            for i, s in enumerate(combo_samples):
                f.write(f'--- Prompt: {prompts[i]} ---\n{s}\n\n')

# Summary table
print(f'\n\n{"=" * 60}')
print('  SWEEP SUMMARY — Review outputs above to pick best combo')
print(f'{"=" * 60}')
print(f'{"#":>3} {"Temp":>6} {"Top-k":>6}  {"First sample preview"}')
print('-' * 70)
for i, r in enumerate(sweep_results, 1):
    preview = r['samples'][0][:60].replace('\n', ' ')
    print(f'{i:>3} {r["temperature"]:>6.1f} {r["top_k"]:>6}  {preview}...')

print(f'\nDefault: temperature=0.8, top_k=100')
print('Update sampling_params.json in Step 10 with your chosen values.')

del model
torch.cuda.empty_cache()
print('\nStep 8 complete!')

## Step 9: PPL Comparison Chart + Results Summary

In [ ]:
import os
import matplotlib.pyplot as plt
os.chdir(WORK_DIR)

# Use ppl_results from Step 6
names = ['M1-Forward', 'M2-Reverse', 'M3-Structured', 'M4-LLaMA']
ppls = [ppl_results.get(n, 0) for n in names]
colors = ['#2196F3', '#FF9800', '#4CAF50', '#9C27B0']

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(names, ppls, color=colors, edgecolor='black', linewidth=0.5)
ax.axhline(y=25.0, color='orange', linestyle='--', linewidth=2, label='PPL = 25.0 threshold')
ax.set_ylabel('Perplexity', fontsize=12)
ax.set_title('ROCStories Perplexity Comparison (Demo 3)', fontsize=14)
ax.legend(fontsize=10)

for bar, ppl in zip(bars, ppls):
    if ppl > 0 and ppl < float('inf'):
        ax.text(bar.get_x() + bar.get_width() / 2., bar.get_height() + 0.5,
                f'{ppl:.1f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('results/ppl_comparison.png', dpi=150)
plt.show()
print('Saved results/ppl_comparison.png')

# Save perplexity table as markdown
with open('results/perplexity_table.md', 'w') as f:
    f.write('# Perplexity Results (Demo 3)\n\n')
    f.write('| Model | PPL | Status |\n')
    f.write('|-------|-----|--------|\n')
    for name in names:
        ppl = ppl_results.get(name, float('inf'))
        status = ''
        if name == 'M1-Forward':
            status = 'PASS' if ppl <= 25.0 else 'FAIL'
        f.write(f'| {name} | {ppl:.2f} | {status} |\n')

    f.write('\n## Hypotheses\n\n')
    m1 = ppl_results.get('M1-Forward', float('inf'))
    m2 = ppl_results.get('M2-Reverse', float('inf'))
    m3 = ppl_results.get('M3-Structured', float('inf'))
    m4 = ppl_results.get('M4-LLaMA', float('inf'))
    if m1 < float('inf') and m2 < float('inf'):
        h1 = 'Supported' if m2 > m1 else 'Not supported'
        f.write(f'- **H1** (M2 PPL > M1 PPL): {h1} (M1={m1:.2f}, M2={m2:.2f})\n')
    if m1 < float('inf') and m3 < float('inf'):
        h2 = 'Supported' if m3 < m1 else 'Not supported'
        f.write(f'- **H2** (M3 PPL < M1 PPL): {h2} (M1={m1:.2f}, M3={m3:.2f})\n')
    if m1 < float('inf') and m4 < float('inf'):
        diff = abs(m4 - m1) / m1 * 100
        if m4 < m1:
            h3 = 'LLaMA better'
        elif diff < 5:
            h3 = 'Approximately equal'
        else:
            h3 = 'GPT-2 better'
        f.write(f'- **H3** (LLaMA vs GPT-2): {h3} (M1={m1:.2f}, M4={m4:.2f}, diff={diff:.1f}%)\n')

print('Saved results/perplexity_table.md')
print('\nStep 9 complete!')

## Step 10: Upload M1 to HuggingFace (SUBMISSION)

In [ ]:
import os
import json
os.chdir(WORK_DIR)

# ============================================================
# CONFIGURE THESE VALUES BEFORE RUNNING
# ============================================================
HF_USERNAME = 'YOUR_HF_USERNAME'   # <-- Change this!
REPO_NAME = 'nanoGPT_hw'

# Sampling parameters — update based on Step 8 sweep results
TEMPERATURE = 0.8     # default, update after reviewing sweep
TOP_K = 100           # default, update after reviewing sweep
MAX_NEW_TOKENS = 200
START_TOKEN = '\n'    # MUST be '\n' for plain text submission (NOT '<S1>')
# ============================================================

# Create sampling_params.json
sampling_params = {
    'temperature': TEMPERATURE,
    'top_k': TOP_K,
    'max_new_tokens': MAX_NEW_TOKENS,
    'start': START_TOKEN
}

with open('sampling_params.json', 'w') as f:
    json.dump(sampling_params, f, indent=2)
print(f'Created sampling_params.json:')
print(json.dumps(sampling_params, indent=2))

# Verify files to upload
upload_files = {
    'ckpt.pt': 'out-m1/ckpt.pt',
    'model.py': 'model.py',
    'sampling_params.json': 'sampling_params.json',
}

print(f'\nFiles to upload to {HF_USERNAME}/{REPO_NAME}:')
for name, path in upload_files.items():
    exists = os.path.exists(path)
    size = os.path.getsize(path) / (1024*1024) if exists else 0
    status = f'OK ({size:.1f} MB)' if exists else 'MISSING'
    print(f'  [{status}] {name} <- {path}')

# Upload to HuggingFace
if HF_USERNAME == 'YOUR_HF_USERNAME':
    print('\n>>> Set HF_USERNAME above before running! <<<')
else:
    from huggingface_hub import HfApi, login

    # Login (will prompt for token if not cached)
    login()

    api = HfApi()
    repo_id = f'{HF_USERNAME}/{REPO_NAME}'

    # Create repo (public)
    try:
        api.create_repo(repo_id=repo_id, repo_type='model', exist_ok=True, private=False)
        print(f'\nRepo created/verified: {repo_id} (PUBLIC)')
    except Exception as e:
        print(f'Repo creation note: {e}')

    # Upload files
    for name, path in upload_files.items():
        if os.path.exists(path):
            print(f'Uploading {name}...')
            api.upload_file(
                path_or_fileobj=path,
                path_in_repo=name,
                repo_id=repo_id,
                repo_type='model',
            )
            print(f'  Uploaded {name}')

    print(f'\nUpload complete!')
    print(f'Repo URL: https://huggingface.co/{repo_id}')
    print(f'\nIMPORTANT: Verify the repo is PUBLIC!')

print('\nStep 10 complete!')